# 07 地震 · 分析样本界定

本阶段只做样本界定和质量清点，**不出时空图、不做目录对照**。主数据用精细清洗结果；外部 USGS 在本文件夹另存并加同一套标记。不改 `07_fine_clean`、深度清洗、原始数据和 `07_external` 原文。


## STEP 1 · 路径

METHOD: 用 `pathlib.Path` 定位本文件夹。优先 `C:\Users\WHB\Desktop\DPW I\07_analysis`，若不存在则回退到当前工作目录或箱子上的同名目录。精细清洗结果与 `07_external` 目录在上一级项目文件夹中查找，并设置必要的回退路径。


In [ ]:
from pathlib import Path
import shutil
import pandas as pd
import numpy as np
from collections import defaultdict

CANDIDATES = [
    Path(r"C:\Users\WHB\Desktop\DPW I\07_analysis"),
    Path.cwd(),
    Path("/workspace/07_analysis"),
]

ROOT = None
for cand in CANDIDATES:
    if cand.exists() and cand.is_dir() and cand.name == "07_analysis":
        ROOT = cand
        break
if ROOT is None:
    for cand in CANDIDATES:
        if cand.exists() and cand.is_dir():
            ROOT = cand
            break
if ROOT is None:
    ROOT = Path("/workspace/07_analysis")
    ROOT.mkdir(parents=True, exist_ok=True)

FINE_CANDIDATES = [
    ROOT.parent / "07_fine_clean",
    Path(r"C:\Users\WHB\Desktop\DPW I\07_fine_clean"),
    Path("/workspace/07_fine_clean"),
]
EXT_CANDIDATES = [
    ROOT.parent / "07_external",
    Path(r"C:\Users\WHB\Desktop\DPW I\07_external"),
    Path("/workspace/07_external"),
]

FINE_ROOT = next((p for p in FINE_CANDIDATES if (p / "cleaned" / "07_earthquake_fine_cleaned.csv").exists()), None)
EXT_ROOT = next((p for p in EXT_CANDIDATES if (p / "catalogs").exists()), None)
if FINE_ROOT is None:
    raise FileNotFoundError("找不到精细清洗结果 07_fine_clean/cleaned/07_earthquake_fine_cleaned.csv")
if EXT_ROOT is None:
    raise FileNotFoundError("找不到 07_external/catalogs")

LOCAL_IN = FINE_ROOT / "cleaned" / "07_earthquake_fine_cleaned.csv"
USGS_2022_IN = FINE_ROOT / "cleaned" / "07_usgs_quakes_fine_cleaned.csv"
USGS_M55_IN = EXT_ROOT / "catalogs" / "07_usgs_quakes_m55_1965_2016.csv"
USGS_2017_IN = EXT_ROOT / "catalogs" / "07_usgs_quakes_m5_2017_2021.csv"
USGS_2024_IN = EXT_ROOT / "catalogs" / "07_usgs_quakes_m5_2024_now.csv"

print("ROOT:", ROOT)
print("FINE_ROOT:", FINE_ROOT)
print("EXT_ROOT:", EXT_ROOT)
print("本地主数据:", LOCAL_IN)
print("USGS 2022-2023 已清洗:", USGS_2022_IN)
print("USGS 1965-2016 M≥5.5:", USGS_M55_IN)
print("USGS 2017-2021 M≥5:", USGS_2017_IN)
print("USGS 2024 至抓取日 M≥5:", USGS_2024_IN)


## STEP 2 · 读取本地精细清洗结果

METHOD: `pd.read_csv` 读取 `07_earthquake_fine_cleaned.csv`。不改列名，不删行，不改 `07_fine_clean` 中的文件。


In [ ]:
df_local = pd.read_csv(LOCAL_IN)
print("已读取:", LOCAL_IN)
print("local shape:", df_local.shape)
print("local columns:", list(df_local.columns))
df_local.head(3)


## STEP 3 · 解析 Date_parsed

METHOD: 读取后的 `Date_parsed` 可能是字符串。用 `pd.to_datetime(..., format="ISO8601", utc=True, errors="coerce")` 写回，统一为 UTC。不改 `Date`、不改 `Time`、不删行。同时把布尔标记列规范为真正的布尔类型。


In [ ]:
def coerce_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    return s.astype(str).str.lower().isin(["true", "1", "yes"])

n_before = len(df_local)
date_before = df_local["Date"].copy()
time_before = df_local["Time"].copy()
df_local["Date_parsed"] = pd.to_datetime(
    df_local["Date_parsed"], format="ISO8601", utc=True, errors="coerce"
)
for col in [
    "is_earthquake", "flag_depth_negative", "flag_coord_invalid",
    "flag_mag_outlier", "flag_depth_over_700", "flag_possible_duplicate",
    "flag_geocode_corrected", "flag_territory_unmatched",
]:
    if col in df_local.columns:
        df_local[col] = coerce_bool(df_local[col])

print("Date_parsed 失败行数:", int(df_local["Date_parsed"].isna().sum()))
print("Date_parsed 范围:", df_local["Date_parsed"].min(), "→", df_local["Date_parsed"].max())
print("行数是否变化:", n_before, "→", len(df_local), "变化量:", len(df_local) - n_before)
print("Date / Time 是否被改写:", (not df_local["Date"].equals(date_before)) or (not df_local["Time"].equals(time_before)))


## STEP 4 · 本地质量清点

METHOD: 统计 `Type` 与各标记的计数，以及震级、深度范围。只观察，不删行、不改值。


In [ ]:
print("Type:")
print(df_local["Type"].value_counts(dropna=False))
print()
print("is_earthquake:")
print(df_local["is_earthquake"].value_counts(dropna=False))
print()
for col in [
    "flag_coord_invalid", "flag_mag_outlier", "flag_depth_over_700",
    "flag_depth_negative", "flag_possible_duplicate",
    "flag_geocode_corrected", "flag_territory_unmatched",
]:
    print(f"{col}: {int(df_local[col].sum())}")
print()
print("Magnitude 范围:", df_local["Magnitude"].min(), "–", df_local["Magnitude"].max(),
      "| 缺失:", int(df_local["Magnitude"].isna().sum()))
print("Depth 范围:", df_local["Depth"].min(), "–", df_local["Depth"].max(),
      "| 缺失:", int(df_local["Depth"].isna().sum()))
print("Depth < 0 仍残留:", int((df_local["Depth"] < 0).sum()))


## STEP 5 · 界定本地主样本

METHOD: 主样本只取 `is_earthquake` 为真的行。不另加震级门槛。非构造事件只计数，不进入主样本，也不从原精细清洗文件中删除。


In [ ]:
n_all = len(df_local)
n_eq = int(df_local["is_earthquake"].sum())
n_non = n_all - n_eq
df_main = df_local.loc[df_local["is_earthquake"]].copy()
print("本地总行数:", n_all)
print("主样本（构造地震）:", n_eq)
print("非构造事件（不进入主样本）:", n_non)
print("非构造 Type 计数:")
print(df_local.loc[~df_local["is_earthquake"], "Type"].value_counts(dropna=False))
print("主样本震级范围:", df_main["Magnitude"].min(), "–", df_main["Magnitude"].max())
print("主样本 Date_parsed 范围:", df_main["Date_parsed"].min(), "→", df_main["Date_parsed"].max())
print("主样本是否另加震级门槛: 否")


## STEP 6 · 空间与震级分析口径

METHOD: 无效坐标不进入后续空间分析，震级异常不进入后续震级统计。即使本数据中这两项为 0 条，也打印规则与计数。此处只登记口径，不删主样本中的行。


In [ ]:
n_coord = int(df_main["flag_coord_invalid"].sum())
n_mag = int(df_main["flag_mag_outlier"].sum())
n_depth700 = int(df_main["flag_depth_over_700"].sum())
print("主样本中 flag_coord_invalid:", n_coord, "（后续空间分析排除）")
print("主样本中 flag_mag_outlier:", n_mag, "（后续震级统计排除）")
print("主样本中 flag_depth_over_700:", n_depth700, "（只标记，不从主样本排除）")
print("可用于空间分析的主样本行数:", len(df_main) - n_coord)
print("可用于震级统计的主样本行数:", len(df_main) - n_mag)


## STEP 7 · 负深度与疑似重复

METHOD: 负深度与疑似重复均保留在主样本中，只报告数量。负深度在精细清洗中已置为空，深度统计时会自然当作缺失。


In [ ]:
n_neg = int(df_main["flag_depth_negative"].sum())
n_dup = int(df_main["flag_possible_duplicate"].sum())
print("主样本中 flag_depth_negative:", n_neg, "（保留；Depth 已为空）")
print("主样本中 flag_possible_duplicate:", n_dup, "（全部保留，只报告）")
if n_neg:
    print()
    print(df_main.loc[df_main["flag_depth_negative"], ["Date", "Time", "Depth", "Magnitude", "flag_depth_negative"]])
print()
print("主样本疑似重复行示例:" if n_dup else "主样本没有疑似重复行。")
if n_dup:
    cols = ["Date", "Time", "Latitude", "Longitude", "Magnitude", "flag_possible_duplicate"]
    print(df_main.loc[df_main["flag_possible_duplicate"], cols].head(8))


## STEP 8 · 登记中国子集

METHOD: 中国子集定义为校正后的 `Country == "CN"`。本步只登记全数据和主样本中的行数，以及主样本的省份分布。不做分省分析。


In [ ]:
df_local["in_china"] = df_local["Country"].astype(str) == "CN"
df_main["in_china"] = df_main["Country"].astype(str) == "CN"
print("全数据中 Country == CN:", int(df_local["in_china"].sum()))
print("主样本中 Country == CN:", int(df_main["in_china"].sum()))
print("主样本中国省份分布（Province）:")
print(df_main.loc[df_main["in_china"], "Province"].value_counts(dropna=False))


## STEP 9 · 导出本地主样本

METHOD: 把主样本写到本文件夹 `07_earthquake_analysis_sample.csv`（`to_csv(index=False)`）。不覆盖精细清洗结果。


In [ ]:
LOCAL_OUT = ROOT / "07_earthquake_analysis_sample.csv"
n_main = len(df_main)
df_main.to_csv(LOCAL_OUT, index=False)
print("wrote:", LOCAL_OUT, "rows:", n_main, "cols:", df_main.shape[1])
print("未覆盖精细清洗文件:", LOCAL_IN.exists(), LOCAL_IN)


## STEP 10 · 登记 USGS 文件口径并读取

METHOD: 同期对照用 `07_external` 中 1965–2016 年 M≥5.5 目录；近年延伸用 2017–2021、已精细清洗的 2022–2023，以及 2024 至抓取日的 M≥5。本步先读取并打印行数、时间与震级范围，先不算对照。


In [ ]:
usgs_files = {
    "m55_1965_2016": USGS_M55_IN,
    "m5_2017_2021": USGS_2017_IN,
    "m5_2022_2023_fine": USGS_2022_IN,
    "m5_2024_now": USGS_2024_IN,
}
usgs_raw = {}
for key, path in usgs_files.items():
    df = pd.read_csv(path)
    usgs_raw[key] = df
    time_col = "Date_parsed" if "Date_parsed" in df.columns else "time"
    mag_col = "mag"
    ts = pd.to_datetime(df[time_col], utc=True, format="ISO8601", errors="coerce")
    if ts.isna().all():
        ts = pd.to_datetime(df[time_col], utc=True, format="mixed", errors="coerce")
    print(f"{key}: {path.name} | {len(df)} 行 × {df.shape[1]} 列 | "
          f"时间 {ts.min()} → {ts.max()} | mag {df[mag_col].min()} – {df[mag_col].max()}")


## STEP 11 · 标记函数（与精细清洗同一套规则）

METHOD: 日期解析为 `pd.to_datetime(..., format="mixed", utc=True, errors="coerce")`，写入 `Date_parsed`。`depth < 0` 加 `flag_depth_negative` 后置为空；`is_earthquake` 为 `type` 忽略大小写后恰好等于 `earthquake`；坐标越界、震级小于 0 或大于 9.5、深度大于 700 只加标记、不改原值。疑似重复：同一文件内，时间圆整到最近分钟，球面距离小于 50 公里且震级差小于 0.3，组内全部加 `flag_possible_duplicate`；USGS 直接用 `Date_parsed`，`Date_parsed` 缺失、坐标无效或震级缺失不配对。不删行。


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def add_duplicate_flag(df, lat_col, lon_col, mag_col):
    out = df.copy()
    out["flag_possible_duplicate"] = False
    ts = pd.to_datetime(out["Date_parsed"], utc=True, errors="coerce")
    minute = ts.dt.round("min")
    valid = ts.notna() & ~out["flag_coord_invalid"]
    tmp = out.loc[valid, [lat_col, lon_col, mag_col]].copy()
    tmp["_minute"] = minute.loc[valid]

    parent = {i: i for i in tmp.index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for _, grp in tmp.groupby("_minute"):
        if len(grp) < 2:
            continue
        idx = grp.index.to_list()
        lats = grp[lat_col].to_numpy(dtype=float)
        lons = grp[lon_col].to_numpy(dtype=float)
        mags = grp[mag_col].to_numpy(dtype=float)
        n = len(grp)
        for i in range(n):
            for j in range(i + 1, n):
                if np.isnan(mags[i]) or np.isnan(mags[j]):
                    continue
                if abs(mags[i] - mags[j]) >= 0.3:
                    continue
                dist = haversine_km(lats[i], lons[i], lats[j], lons[j])
                if dist < 50:
                    union(idx[i], idx[j])

    comps = defaultdict(list)
    for i in tmp.index:
        comps[find(i)].append(i)
    n_groups = 0
    n_rows = 0
    for members in comps.values():
        if len(members) >= 2:
            out.loc[members, "flag_possible_duplicate"] = True
            n_groups += 1
            n_rows += len(members)
    return out, n_groups, n_rows


def add_usgs_flags(df):
    out = df.copy()
    n0 = len(out)
    out["Date_parsed"] = pd.to_datetime(out["time"], format="mixed", utc=True, errors="coerce")
    out["flag_depth_negative"] = out["depth"] < 0
    out["depth"] = out["depth"].mask(out["flag_depth_negative"])
    out["is_earthquake"] = out["type"].astype(str).str.lower() == "earthquake"
    out["flag_coord_invalid"] = (
        (out["latitude"] < -90) | (out["latitude"] > 90)
        | (out["longitude"] < -180) | (out["longitude"] > 180)
    )
    out["flag_mag_outlier"] = (out["mag"] < 0) | (out["mag"] > 9.5)
    out["flag_depth_over_700"] = out["depth"] > 700
    out, n_dup_g, n_dup_r = add_duplicate_flag(out, "latitude", "longitude", "mag")
    assert len(out) == n0
    summary = {
        "rows": n0,
        "date_fail": int(out["Date_parsed"].isna().sum()),
        "date_min": out["Date_parsed"].min(),
        "date_max": out["Date_parsed"].max(),
        "is_earthquake": int(out["is_earthquake"].sum()),
        "non_earthquake": int((~out["is_earthquake"]).sum()),
        "flag_depth_negative": int(out["flag_depth_negative"].sum()),
        "flag_coord_invalid": int(out["flag_coord_invalid"].sum()),
        "flag_mag_outlier": int(out["flag_mag_outlier"].sum()),
        "flag_depth_over_700": int(out["flag_depth_over_700"].sum()),
        "dup_groups": n_dup_g,
        "flag_possible_duplicate": n_dup_r,
        "mag_min": out["mag"].min(),
        "mag_max": out["mag"].max(),
    }
    return out, summary

print("标记函数已就绪。")


## STEP 12 · 为外部 USGS 目录加标记

METHOD: 对 1965–2016 年 M≥5.5、2017–2021 年 M≥5、2024 至抓取日 M≥5 三份外部目录分别加同一套标记。不删行，不改 `07_external` 原文。


In [ ]:
usgs_flagged = {}
usgs_summaries = {}
for key in ["m55_1965_2016", "m5_2017_2021", "m5_2024_now"]:
    flagged, summary = add_usgs_flags(usgs_raw[key])
    usgs_flagged[key] = flagged
    usgs_summaries[key] = summary
    print(f"—— {key} ——")
    for k, v in summary.items():
        print(f"  {k}: {v}")
    print()


## STEP 13 · 登记已精细清洗的 USGS 2022–2023

METHOD: 2022–2023 年目录已在精细清洗中加过同一套标记，本步直接使用，不重跑标记。只规范布尔列并解析 `Date_parsed`，不改原清洗文件。


In [ ]:
df_usgs_2022 = usgs_raw["m5_2022_2023_fine"].copy()
df_usgs_2022["Date_parsed"] = pd.to_datetime(
    df_usgs_2022["Date_parsed"], format="ISO8601", utc=True, errors="coerce"
)
for col in [
    "is_earthquake", "flag_depth_negative", "flag_coord_invalid",
    "flag_mag_outlier", "flag_depth_over_700", "flag_possible_duplicate",
]:
    if col in df_usgs_2022.columns:
        df_usgs_2022[col] = coerce_bool(df_usgs_2022[col])
usgs_flagged["m5_2022_2023_fine"] = df_usgs_2022
usgs_summaries["m5_2022_2023_fine"] = {
    "rows": len(df_usgs_2022),
    "date_fail": int(df_usgs_2022["Date_parsed"].isna().sum()),
    "date_min": df_usgs_2022["Date_parsed"].min(),
    "date_max": df_usgs_2022["Date_parsed"].max(),
    "is_earthquake": int(df_usgs_2022["is_earthquake"].sum()),
    "non_earthquake": int((~df_usgs_2022["is_earthquake"]).sum()),
    "flag_depth_negative": int(df_usgs_2022["flag_depth_negative"].sum()),
    "flag_coord_invalid": int(df_usgs_2022["flag_coord_invalid"].sum()),
    "flag_mag_outlier": int(df_usgs_2022["flag_mag_outlier"].sum()),
    "flag_depth_over_700": int(df_usgs_2022["flag_depth_over_700"].sum()),
    "dup_groups": "已有精细清洗，本步不重算组数",
    "flag_possible_duplicate": int(df_usgs_2022["flag_possible_duplicate"].sum()),
    "mag_min": df_usgs_2022["mag"].min(),
    "mag_max": df_usgs_2022["mag"].max(),
}
print("使用已精细清洗的 2022-2023，不重跑标记。")
for k, v in usgs_summaries["m5_2022_2023_fine"].items():
    print(f"  {k}: {v}")


## STEP 14 · 导出 USGS 标记副本与质量汇总表

METHOD: 在本文件夹写出 USGS 标记副本，以及一份质量清点表。不覆盖 `07_external` 与 `07_fine_clean` 中的文件。


In [ ]:
USGS_OUT = {
    "m55_1965_2016": ROOT / "07_usgs_m55_1965_2016_flagged.csv",
    "m5_2017_2021": ROOT / "07_usgs_m5_2017_2021_flagged.csv",
    "m5_2022_2023_fine": ROOT / "07_usgs_m5_2022_2023_flagged.csv",
    "m5_2024_now": ROOT / "07_usgs_m5_2024_now_flagged.csv",
}
for key, path in USGS_OUT.items():
    usgs_flagged[key].to_csv(path, index=False)
    print("wrote:", path.name, "rows:", len(usgs_flagged[key]))

rows = [
    {
        "dataset": "local_fine_cleaned",
        "role": "本地全数据（精细清洗）",
        "rows": n_all,
        "is_earthquake": n_eq,
        "non_earthquake": n_non,
        "in_china": int(df_local["in_china"].sum()),
        "flag_coord_invalid": int(df_local["flag_coord_invalid"].sum()),
        "flag_mag_outlier": int(df_local["flag_mag_outlier"].sum()),
        "flag_depth_over_700": int(df_local["flag_depth_over_700"].sum()),
        "flag_depth_negative": int(df_local["flag_depth_negative"].sum()),
        "flag_possible_duplicate": int(df_local["flag_possible_duplicate"].sum()),
        "date_min": str(df_local["Date_parsed"].min()),
        "date_max": str(df_local["Date_parsed"].max()),
    },
    {
        "dataset": "local_main_sample",
        "role": "本地主样本（构造地震）",
        "rows": n_main,
        "is_earthquake": n_main,
        "non_earthquake": 0,
        "in_china": int(df_main["in_china"].sum()),
        "flag_coord_invalid": n_coord,
        "flag_mag_outlier": n_mag,
        "flag_depth_over_700": n_depth700,
        "flag_depth_negative": n_neg,
        "flag_possible_duplicate": n_dup,
        "date_min": str(df_main["Date_parsed"].min()),
        "date_max": str(df_main["Date_parsed"].max()),
    },
]
role_map = {
    "m55_1965_2016": "USGS 同期对照（M≥5.5，1965–2016）",
    "m5_2017_2021": "USGS 近年延伸（M≥5，2017–2021）",
    "m5_2022_2023_fine": "USGS 近年延伸（M≥5，2022–2023，已精细清洗）",
    "m5_2024_now": "USGS 近年延伸（M≥5，2024 至抓取日）",
}
for key, summary in usgs_summaries.items():
    rows.append({
        "dataset": key,
        "role": role_map[key],
        "rows": summary["rows"],
        "is_earthquake": summary["is_earthquake"],
        "non_earthquake": summary["non_earthquake"],
        "in_china": pd.NA,
        "flag_coord_invalid": summary["flag_coord_invalid"],
        "flag_mag_outlier": summary["flag_mag_outlier"],
        "flag_depth_over_700": summary["flag_depth_over_700"],
        "flag_depth_negative": summary["flag_depth_negative"],
        "flag_possible_duplicate": summary["flag_possible_duplicate"],
        "date_min": str(summary["date_min"]),
        "date_max": str(summary["date_max"]),
    })
quality = pd.DataFrame(rows)
QUALITY_OUT = ROOT / "07_analysis_sample_quality.csv"
quality.to_csv(QUALITY_OUT, index=False)
print("wrote:", QUALITY_OUT)
print()
print(quality.to_string(index=False))


## STEP 15 · 回读检查，确认原文未被改动

METHOD: 重新读回刚写出的 CSV，核对行数；并确认精细清洗文件与 `07_external` 原文路径仍存在，且不是本步的输出路径。


In [ ]:
local_chk = pd.read_csv(LOCAL_OUT)
print("回读本地主样本:", local_chk.shape, "期望", (n_main, df_main.shape[1]))
print("主样本行数一致:", len(local_chk) == n_main)
print("主样本均为构造地震:", bool(coerce_bool(local_chk["is_earthquake"]).all()))
print("含 in_china 列:", "in_china" in local_chk.columns)

for key, path in USGS_OUT.items():
    chk = pd.read_csv(path)
    print(f"回读 {path.name}: {chk.shape} 期望行数 {len(usgs_flagged[key])}, 一致: {len(chk) == len(usgs_flagged[key])}")

originals = [LOCAL_IN, USGS_2022_IN, USGS_M55_IN, USGS_2017_IN, USGS_2024_IN]
outputs = [LOCAL_OUT, QUALITY_OUT, *USGS_OUT.values()]
print()
print("原文仍在:", all(p.exists() for p in originals))
print("输出没有覆盖原文:", set(originals).isdisjoint(set(outputs)))
for p in originals:
    print("  keep:", p)
